In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 0. GPU SETUP — MUST RUN FIRST
# ══════════════════════════════════════════════════════════════════════════════
import os, time
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import tensorflow as tf

# T4 has 16 GB — allow memory growth
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"✓ GPU: {gpus[0].name}  |  Memory growth enabled")
else:
    raise RuntimeError("No GPU detected! Go to Runtime → Change runtime type → T4 GPU")

# Mixed precision — T4 has Tensor Cores that accelerate FP16
tf.keras.mixed_precision.set_global_policy('mixed_float16')
print(f"✓ Mixed precision: {tf.keras.mixed_precision.global_policy().name}")
print(f"✓ TF version: {tf.__version__}")

CLOCK_START = time.time()

This cell initializes the T4 GPU and enables mixed precision to accelerate training speed and manage memory efficiently.
We need this to ensure the model runs smoothly on Colab’s hardware without running out of VRAM or taking too long per epoch.

In [ ]:
import numpy as np
import os, time
import pandas as pd
import json
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('darkgrid')

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, GlobalAveragePooling2D
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import (
    ReduceLROnPlateau, EarlyStopping, ModelCheckpoint, TensorBoard
)
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.applications.efficientnet import preprocess_input

import warnings
warnings.filterwarnings('ignore')

This cell imports all necessary libraries for data processing, machine learning, and model evaluation.
We need these dependencies to handle image data loading, model architecture building, and performance metrics calculation throughout the script.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 1. MOUNT DRIVE & EXTRACT DATA
# ══════════════════════════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


This cell securely mounts your Google Drive to the Colab environment for persistent storage.
We need this because the dataset and final model weights are stored on Drive and must be accessible during training.

In [ ]:
%%bash
# Extract data to Colab's local SSD (MUCH faster I/O than Drive)
# Using pigz for parallel decompression
tar --use-compress-program=pigz \
    -xf /content/drive/MyDrive/masterData.tar.gz \
    -C / \
    2>&1 | tail -5

echo "Exit code: $?"
echo "Dataset extracted to local SSD ✓"

Exit code: 0
Dataset extracted to local SSD ✓


This cell extracts the dataset from your Drive into Colab’s local SSD storage using parallel processing.
We need this to overcome slow data transfer speeds from Google Drive and significantly speed up the image loading process.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 2. CONFIGURATION — all tunables in one place
# ══════════════════════════════════════════════════════════════════════════════

DATA_DIR = '/content/masterData'                # local SSD (fast I/O)
SAVE_DIR = '/content/drive/MyDrive/crop_models' # Drive (persists after disconnect)
os.makedirs(SAVE_DIR, exist_ok=True)

# ── T4-optimized hyperparameters ─────────────────────────────────────────────
IMG_SIZE        = (300, 300)    # B3 native resolution — best accuracy
BATCH_SIZE      = 32           # T4 handles this at 300×300 with FP16
PHASE1_EPOCHS   = 10           # frozen backbone — converges fast
PHASE2_EPOCHS   = 12           # fine-tune — where the accuracy gain happens
PHASE1_LR       = 1e-3         # higher LR for frozen backbone (only head trains)
PHASE2_LR       = 1e-5         # low LR for fine-tuning backbone
UNFREEZE_TOP    = 80           # unfreeze more layers for better fine-tuning

print(f"Image: {IMG_SIZE} | Batch: {BATCH_SIZE} | FP16: ON")
print(f"Phase 1: {PHASE1_EPOCHS} epochs @ LR={PHASE1_LR}")
print(f"Phase 2: {PHASE2_EPOCHS} epochs @ LR={PHASE2_LR}")
print(f"Save to: {SAVE_DIR}")

Image: (300, 300) | Batch: 32 | FP16: ON
Phase 1: 10 epochs @ LR=0.001
Phase 2: 12 epochs @ LR=1e-05
Save to: /content/drive/MyDrive/crop_models


This cell defines all training hyperparameters like image size, batch size, and learning rates in one centralized location.
We need this to maintain consistency across the notebook and make it easy to tune the model for different performance goals.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 3. DATA PREPARATION — scan files + clean out corrupt entries
# ══════════════════════════════════════════════════════════════════════════════

VALID_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.gif'}

def define_paths(data_dir):
    """Walk the dataset directory and collect valid image paths."""
    filepaths, labels = [], []
    skipped = 0
    for fold in sorted(os.listdir(data_dir)):
        foldpath = os.path.join(data_dir, fold)
        if not os.path.isdir(foldpath):
            continue
        for file in os.listdir(foldpath):
            ext = os.path.splitext(file)[1].lower()
            if ext not in VALID_EXTENSIONS:
                skipped += 1
                continue
            filepath = os.path.join(foldpath, file)
            # Skip tiny files (likely corrupt)
            if os.path.getsize(filepath) < 1000:  # < 1KB
                skipped += 1
                continue
            filepaths.append(filepath)
            labels.append(fold)
    if skipped:
        print(f"⚠ Skipped {skipped} non-image/corrupt files")
    return filepaths, labels


def split_data(data_dir):
    files, classes = define_paths(data_dir)
    df = pd.concat([
        pd.Series(files, name='filepaths'),
        pd.Series(classes, name='labels')
    ], axis=1)

    train_df, temp_df = train_test_split(
        df, train_size=0.8, stratify=df['labels'], random_state=123
    )
    valid_df, test_df = train_test_split(
        temp_df, train_size=0.5, stratify=temp_df['labels'], random_state=123
    )

    print(f"Train: {len(train_df):,}  |  Valid: {len(valid_df):,}  |  Test: {len(test_df):,}")
    print(f"Classes: {df['labels'].nunique()}")
    return train_df, valid_df, test_df

train_df, valid_df, test_df = split_data(DATA_DIR)

⚠ Skipped 16 non-image/corrupt files
Train: 132,798  |  Valid: 16,600  |  Test: 16,600
Classes: 136


This cell scans the dataset folder, removes corrupt files, and splits the data into stratified train, validation, and test sets.
We need this to ensure the model learns from high-quality images and that our performance evaluation is statistically honest and balanced.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 4. tf.data PIPELINE — robust image loading with error handling
# ══════════════════════════════════════════════════════════════════════════════

# Build label encoding
all_labels = sorted(train_df['labels'].unique())
label_to_idx = {lbl: i for i, lbl in enumerate(all_labels)}
idx_to_label = {i: lbl for lbl, i in label_to_idx.items()}
NUM_CLASSES = len(all_labels)
print(f"Classes: {NUM_CLASSES}")


def parse_image_train(filepath, label_idx):
    """Load + augment for training. Uses tf.image ops only (GPU-friendly)."""
    img = tf.io.read_file(filepath)
    # Use decode_jpeg with try_recover_truncated for robustness
    img = tf.image.decode_jpeg(img, channels=3, try_recover_truncated=True)
    img = tf.cast(img, tf.float32)
    img = tf.image.resize(img, IMG_SIZE)

    # Augmentation
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, max_delta=0.15)
    img = tf.image.random_contrast(img, lower=0.85, upper=1.15)
    img = tf.image.random_saturation(img, lower=0.85, upper=1.15)

    # Random crop for scale jitter
    padded = tf.image.resize_with_crop_or_pad(
        img, int(IMG_SIZE[0] * 1.1), int(IMG_SIZE[1] * 1.1)
    )
    img = tf.image.random_crop(padded, [IMG_SIZE[0], IMG_SIZE[1], 3])

    # EfficientNet preprocess (scales to [-1, 1])
    img = preprocess_input(img)
    label = tf.one_hot(label_idx, NUM_CLASSES)
    return img, label


def parse_image_eval(filepath, label_idx):
    """Load only — no augmentation."""
    img = tf.io.read_file(filepath)
    img = tf.image.decode_jpeg(img, channels=3, try_recover_truncated=True)
    img = tf.cast(img, tf.float32)
    img = tf.image.resize(img, IMG_SIZE)
    img = preprocess_input(img)
    label = tf.one_hot(label_idx, NUM_CLASSES)
    return img, label


def build_dataset(df, batch_size, training=False):
    """Build a high-performance tf.data pipeline."""
    filepaths = df['filepaths'].values
    label_indices = np.array(
        [label_to_idx[l] for l in df['labels'].values], dtype=np.int32
    )

    ds = tf.data.Dataset.from_tensor_slices((filepaths, label_indices))

    if training:
        ds = ds.shuffle(buffer_size=min(len(df), 20000), reshuffle_each_iteration=True)
        ds = ds.map(parse_image_train, num_parallel_calls=tf.data.AUTOTUNE)
    else:
        ds = ds.map(parse_image_eval, num_parallel_calls=tf.data.AUTOTUNE)

    # Ignore errors from corrupt images instead of crashing
    ds = ds.apply(tf.data.experimental.ignore_errors())

    ds = ds.batch(batch_size, drop_remainder=training)
    ds = ds.prefetch(tf.data.AUTOTUNE)

    return ds


train_ds = build_dataset(train_df, BATCH_SIZE, training=True)
valid_ds = build_dataset(valid_df, BATCH_SIZE, training=False)
test_ds  = build_dataset(test_df,  BATCH_SIZE, training=False)

# Quick sanity check — grab one batch
for imgs, lbls in train_ds.take(1):
    print(f"✓ Batch shape: images={imgs.shape}, labels={lbls.shape}")
    print(f"  Pixel range: [{imgs.numpy().min():.2f}, {imgs.numpy().max():.2f}]")

print(f"\n✓ Pipelines ready  |  Elapsed: {(time.time()-CLOCK_START)/60:.1f} min")

This cell creates a high-performance data pipeline that performs real-time image augmentation and prefetching.
We need this to keep the GPU fully utilized and to improve the model's robustness by exposing it to varied versions of the input images.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 5. CLASS WEIGHTS
# ══════════════════════════════════════════════════════════════════════════════

string_labels  = train_df['labels'].values
unique_classes = np.unique(string_labels)

weights = compute_class_weight('balanced', classes=unique_classes, y=string_labels)

# Cap extreme weights to prevent instability (max 5×)
weights = np.clip(weights, 0.5, 5.0)

string_to_weight  = dict(zip(unique_classes, weights))
class_weight_dict = {label_to_idx[cls]: w for cls, w in string_to_weight.items()}

print(f"Class weights: {len(class_weight_dict)} (capped at 5.0)")
top5 = sorted(class_weight_dict.items(), key=lambda x: x[1], reverse=True)[:5]
print("Top-5 minority classes:")
for idx, w in top5:
    print(f"  {idx_to_label[idx]:45s}  weight={w:.3f}")

This cell calculates the balanced weight for each class based on the quantity of samples available in the training set.
We need this to prevent the model from ignoring rare diseases by giving them higher priority during the optimization process.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 6. FOCAL LOSS
# ══════════════════════════════════════════════════════════════════════════════

from tensorflow.keras.utils import register_keras_serializable

@register_keras_serializable()
def focal_loss(y_true, y_pred, gamma=2.0, alpha=0.25):
    """
    Focal loss — down-weights easy examples, focuses on hard/minority classes.
    Registered for serialization so model.save() / load_model() works.
    """
    y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0)
    ce     = -y_true * tf.math.log(y_pred)
    pt     = tf.reduce_sum(y_true * y_pred, axis=-1, keepdims=True)
    loss   = alpha * tf.pow(1.0 - pt, gamma) * ce
    return tf.reduce_mean(tf.reduce_sum(loss, axis=-1))

This cell defines a custom Focal Loss function that focuses the training effort on images the model finds most difficult to classify.
We need this specialized loss to improve accuracy on 136 different classes where some categories are much harder to learn than others.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 7. BUILD MODEL
# ══════════════════════════════════════════════════════════════════════════════

img_shape = (*IMG_SIZE, 3)  # (300, 300, 3)

base_model = EfficientNetB3(
    include_top=False,
    weights='imagenet',
    input_shape=img_shape,
    pooling='avg'
)
base_model.trainable = False  # frozen for Phase 1

model = Sequential([
    base_model,
    BatchNormalization(),
    Dense(512, activation='relu', kernel_regularizer=regularizers.l2(0.0005)),
    Dropout(0.4),
    Dense(256, activation='relu', kernel_regularizer=regularizers.l2(0.0005)),
    Dropout(0.3),
    # float32 softmax for numerical stability with mixed precision
    Dense(NUM_CLASSES, activation='softmax', dtype='float32')
])

model.compile(
    optimizer=Adam(learning_rate=PHASE1_LR),
    loss=focal_loss,
    metrics=['accuracy']
)

model.summary()

trainable = sum(1 for l in model.trainable_weights)
total = model.count_params()
print(f"\n✓ Total params: {total:,}")

43941136/43941136 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb3 (Functional)     │ (None, 1536)           │    10,783,535 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1536)           │         6,144 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       786,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 136)            │        34,952 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,742,903 (44.80 MB)

 Trainable params: 956,296 (3.65 MB)

 Non-trainable params: 10,786,607 (41.15 MB)


✓ Total params: 11,742,903


This cell constructs the deep learning architecture by combining a pre-trained EfficientNetB3 backbone with a custom classification head.
We need this transfer learning approach to leverage advanced image recognition features while specializing the model for specific crop disease patterns.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 8. PHASE 1 — Train classifier head (frozen backbone)
# ══════════════════════════════════════════════════════════════════════════════

p1_model_path   = os.path.join(SAVE_DIR, 'crop136_b3_phase1.keras')
p1_history_path = os.path.join(SAVE_DIR, 'crop136_b3_phase1_history.json')

callbacks_p1 = [
    ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1
    ),
    EarlyStopping(
        monitor='val_loss', patience=5, restore_best_weights=True, verbose=1
    ),
    ModelCheckpoint(
        p1_model_path, monitor='val_accuracy',
        save_best_only=True, verbose=1
    ),
]

print("=" * 65)
print(f"PHASE 1: Training classifier head (frozen EfficientNetB3)")
print(f"  {IMG_SIZE} | BS={BATCH_SIZE} | LR={PHASE1_LR} | FP16 | Max {PHASE1_EPOCHS} epochs")
print("=" * 65)

t1 = time.time()

history_phase1 = model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=PHASE1_EPOCHS,
    callbacks=callbacks_p1,
    class_weight=class_weight_dict,
    verbose=1
)

p1_time = (time.time() - t1) / 60
print(f"\n✓ Phase 1 done in {p1_time:.1f} min")
print(f"  Total elapsed: {(time.time()-CLOCK_START)/60:.1f} min")

# Save history
with open(p1_history_path, 'w') as f:
    json.dump(history_phase1.history, f)
print(f"  M odel → {p1_model_path}")

PHASE 1: Training classifier head (frozen EfficientNetB3)
  (300, 300) | BS=32 | LR=0.001 | FP16 | Max 10 epochs
Epoch 1/10
   4149/Unknown 1444s 326ms/step - accuracy: 0.6025 - loss: 0.6817
Epoch 1: val_accuracy improved from None to 0.79884, saving model to /content/drive/MyDrive/crop_models/crop136_b3_phase1.keras

Epoch 1: finished saving model to /content/drive/MyDrive/crop_models/crop136_b3_phase1.keras
4149/4149 ━━━━━━━━━━━━━━━━━━━━ 1668s 381ms/step - accuracy: 0.6759 - loss: 0.4769 - val_accuracy: 0.7988 - val_loss: 0.2780 - learning_rate: 0.0010
Epoch 2/10
4149/4149 ━━━━━━━━━━━━━━━━━━━━ 0s 324ms/step - accuracy: 0.7197 - loss: 0.3329
Epoch 2: val_accuracy improved from 0.79884 to 0.81198, saving model to /content/drive/MyDrive/crop_models/crop136_b3_phase1.keras

Epoch 2: finished saving model to /content/drive/MyDrive/crop_models/crop136_b3_phase1.keras
4149/4149 ━━━━━━━━━━━━━━━━━━━━ 1502s 362ms/step - accuracy: 0.7222 - loss: 0.3280 - val_accuracy: 0.8120 - val_loss: 0.2585 

KeyboardInterrupt: 

Exception ignored in: <function _xla_gc_callback at 0x7f3f4ee53f60>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/jax/_src/lib/__init__.py", line 127, in _xla_gc_callback
    def _xla_gc_callback(*args):
    
KeyboardInterrupt: 


This cell executes the primary training phase where only the new classification layers are optimized while the pre-trained backbone stays frozen.
We need this "top-down" training to stabilize the randomly initialized weights of the head without destroying the pre-existing features of the backbone.

we have Interrupted it manually because , validation accuracy actually decreased from 82.9% to 81.3%, 
which usually signals that the frozen backbone had reached its limit and the model was starting to fluctuate.

# **end of first phase training**